In [1]:
import re, sys, os
from pathlib import Path
import torch
import numpy as np
repo_start = f'../'
sys.path.append(repo_start)

from modules.utils.imports import *
from modules.binn_eql.model_wrapper_2d import model_wrapper
from modules.binn_eql.build_binn_eql_net import BINN
# from modules.binn_eql_hypernet.build_binn_eql_net import BINN
from modules.utils.format_data import format_u_array_to_training_data

In [2]:
def extract_final_loss(filepath, loss_types=['pde', 'gls', 'reg']):
    """
    Reads the given file and extracts the validation loss (as a float)
    from the line containing the word 'Elapsed'.
    """
    # Regular expression to capture the validation loss
    # This regex looks for "Val loss = " followed by a floating point number (possibly in scientific notation)
    # val_loss_pattern = re.compile(r"Val loss = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
    pde_pattern = re.compile(r"Val PDE = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
    gls_pattern = re.compile(r"Val GLS = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
    reg_pattern = re.compile(r"Val Reg = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")

    target_line = None
    last_line = ""

    with open(filepath, 'r') as file:             
        for line in file:
            last_line = line  # Keep track of the current line
            
            if "Elapsed" in line:
                target_line = line
                break  # Stop reading the file, we found what we need

    # If the loop finished without finding "Elapsed", fallback to the very last line
    if target_line is None:
        target_line = last_line

    # Now apply your extraction logic to whichever line was chosen
    if target_line.strip():  # Ensure the line isn't completely empty
        pde = float(pde_pattern.search(target_line).group(1))
        gls = float(gls_pattern.search(target_line).group(1))
        reg = float(reg_pattern.search(target_line).group(1))

    loss = 0
    
    if 'pde' in loss_types: loss += pde
    if 'gls' in loss_types: loss += gls
    if 'reg' in loss_types: loss += reg
    
    return loss
                
def process_directory(parent_dir, loss_types):
    """
    For each subdirectory under parent_dir, finds slurm files with the pattern "slurm-####.out",
    selects the one with the lower number, extracts the validation loss, and prints the subdirectory path
    alongside the extracted loss.
    """
    results = {}
    # Use pathlib to iterate over subdirectories
    for subdir in Path(parent_dir).iterdir():
        if subdir.is_dir():
            # Find files matching pattern "slurm-*.out"
            log_files = list(subdir.glob('training.log'))

            if not log_files:
                # print(f"No slurm files found in {subdir}")
                results[str(subdir)] = None
                continue
            
            # Extract numeric part from filename (assumes naming: slurm-####.out)
            def extract_number(file_path):
                match = re.search(r"slurm-(\d+)\.out", file_path.name)
                return int(match.group(1)) if match else float('inf')
            
            # Sort files by the extracted number
            log_files.sort(key=extract_number)

            # Take the first file (lowest number)
            first_file = log_files[0]
            val_loss = extract_final_loss(first_file, loss_types)
            
            if val_loss is not None:
                results[str(subdir)] = val_loss
            else:
                # print(f"Could not extract validation loss from file {first_file} in {subdir}")
                results[str(subdir)] = None
    
    return results

In [11]:
# Get config data
idx = 4
dir_list = ['/hpc/home/nsmyers1/projects/kinetic_binns/development/paper_equations/2_longer_phases/hill_hill_du_0.01_dv_1_a_4_b_2_k1_0.1_n1_3_k2_0.5_n2_2',
            '/hpc/home/nsmyers1/projects/kinetic_binns/development/paper_equations/2_longer_phases/hill_hill_du_0.01_dv_1_a_2_b_2_k1_0.1_n1_3_k2_0.01_n2_2',
            '/hpc/home/nsmyers1/projects/kinetic_binns/development/paper_equations/2_longer_phases/hill_poly_du_0.01_dv_1_a_4_b_2_k_0.5_n_3',
            '/hpc/home/nsmyers1/projects/kinetic_binns/development/paper_equations/2_longer_phases/poly_hill_du_0.01_dv_1_a_1_b_1_k_0.1_n_2',
            '/hpc/home/nsmyers1/projects/kinetic_binns/development/paper_equations/2_longer_phases/poly_poly_du_0.01_dv_1_a_4_b_2']
parent_directory = dir_list[idx]

# parent_directory = '/hpc/home/nsmyers1/projects/kinetic_binns/development/custom_equation/23_normal_uvmlp_learnable_b/du_0.01_dv_1.0_a_1.0_b_1.0_k_0.1'
# parent_directory = '/hpc/home/nsmyers1/projects/kinetic_binns/development/custom_equation/23_normal_uvmlp_learnable_b/du_0.01_dv_1.0_a_4.0_b_4.0_k_0.05'

repeat_dir = os.listdir(parent_directory)[0]
config_path = Path(f'{parent_directory}/{repeat_dir}') / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# 2. Load variable from JSON
training_data_path = config['training_data_path']
batch_size = config['batch_size']
species = config['species']           
dimensions = config['dimensions']     
epsilon = config['epsilon']           
points = config['points']             
params = config['params']             
diff_coeffs = config['diff_coeffs']   

duplicates = config['duplicates']
degree = config['degree']
pde_weight = config['pde_weight']
l0_weight = config['l0_weight']
# warm_up = config['warm_up']
lux_tax = config['lux_tax']
param_bounds = config['param_bounds']
training_data = torch.load(training_data_path)['training_data']

# NEED TO FIX SIMPLE COEFFICIENT SWAP DURING FINE TUNING, DOESN'T WORK FOR DECREASING HILL FUNCTIONS

In [12]:
loss_types = ['reg', 'pde']

losses = process_directory(parent_directory, loss_types)

# Sort the results by validation loss (lowest first)
sorted_losses = sorted(losses.items(), key=lambda item: (item[1] is None, item[1]))

# Print the sorted results
for dir_name, val_loss in sorted_losses:
    if val_loss:
        print(f"\nDirectory: {dir_name.split('/')[-1]}")
        print(f"Validation Loss: {val_loss}")
        
        binn = BINN(
            dimensions=dimensions,
            species=species, 
            train_data=training_data, 
            duplicates=duplicates,
            diff_coeffs=diff_coeffs,
            degree=degree,
            param_bounds=param_bounds)

        binn.to('cpu')

        parameters = binn.parameters()

        opt = torch.optim.Adam(parameters, lr=0.001)

        model = model_wrapper(
            model=binn,
            optimizer=opt,
            loss=binn.loss,
            dir_name=dir_name,
            save_name=f'{dir_name}/binn')
                
        model.load(f"{dir_name}/binn_best_val_model", device='cpu')
        # model.model.prune(thresh=5)
        # Print equation
        fn = f'{dir_name}/equation.txt'
        for term in model.model.generate_equation():
            print(f'{term}')
        
        model.model.fine_tune_eql(threshold=0.01, epsilon=0.1)
        for term in model.model.generate_equation():
            print(f'{term}')
            
        if not model.model.diff_coeffs:          
            print(f'{[D.item() for D in model.model.diffusion_fitter()]}\n')



Directory: binn_eql_batch_50000_points_100_repeat_5
Validation Loss: 13.147200000000002
1.443 * u^2 * v
-0.861 * u
-0.666 * u
1.471 * u^2 * v
-0.425 * u
0.997 * u^2 * v
Fine-tuning committed.
-1.952 * u
3.911 * u^2 * v

Directory: binn_eql_batch_50000_points_100_repeat_2
Validation Loss: 15.3842
2.010 * u^2 * v
-1.262 * u^1.023 / (1 + 0.397 * u^1.023)
-0.206 * u^2.398 / (1 + 0.024 * u^2.398)
0.796 * v * u^2.588 / (1 + 0.003 * u^2.588)
-0.447 * v * (1 / 0.035 - u^2.748 / (1 + 0.035 * u^2.748))
-2.072 * u * (1 / 2.652 - v^1.089 / (1 + 2.652 * v^1.089))
1.022 * v * (1 / 0.080 - u^1.000 / (1 + 0.080 * u^1.000))
Merging Duplicate Hills: 45 and 53 (Dist: 0.0665)
Moving weight 1.0223 directly to Poly 1, max_denom=1.83, n=1.000
Fine-tuning committed.
1.022 * v
2.010 * u^2 * v
-1.467 * u^1.044 / (1 + 0.278 * u^1.044)
0.796 * v * u^2.588 / (1 + 0.003 * u^2.588)
-0.447 * v * (1 / 0.035 - u^2.748 / (1 + 0.035 * u^2.748))
-2.072 * u * (1 / 2.652 - v^1.089 / (1 + 2.652 * v^1.089))

Directory: binn_

In [32]:
# Load model
binn = BINN(
    dimensions=2,
    species=2, 
    train_data=training_data, 
    diff_coeffs=[0.01, 1],
    # uv_layers=[256, 256, 256, 256, 2],
    # diff_coeffs=(),
    duplicates=5)

binn.to('cpu')

parameters = binn.parameters()

opt = torch.optim.Adam(parameters, lr=0.001)

model = model_wrapper(
    model=binn,
    optimizer=opt,
    loss=binn.loss,
    dir_name=dir_name,
    save_name=f'{dir_name}/binn')

model.load(f"/work/users/s/m/smyersn/elston/projects/kinetics_binns/development/binn_eql_net/runs/pos_feedback/76_more_l0_loss/a_1.0_b_1_k_0.01/binn_eql_l0_2_duplicates_5_warm_up_20000_lux_tax_1_repeat_3/binn_best_val_model", device='cpu')

# DIFFUSION COEFFICIENTS

In [33]:
parent_directory = '/hpc/home/nsmyers1/projects/kinetic_binns/development/diff_coeffs/11_32_pt_precision/du_0.01_dv_1.0_a_1.0_b_1.0_k_0.01'
loss_types = ['pde']

losses = process_directory(parent_directory, loss_types)

# Sort the results by validation loss (lowest first)
sorted_losses = sorted(losses.items(), key=lambda item: (item[1] is None, item[1]))

# Print the sorted results
for dir_name, val_loss in sorted_losses:
    if val_loss:
        print(f"\nDirectory: {dir_name.split('/')[-1]}")
        print(f"Validation Loss: {val_loss}")
        
        # Load model
        binn = BINN(
            dimensions=2,
            species=2, 
            train_data=training_data, 
            diff_coeffs=[],
            # uv_layers=[256, 256, 256, 256, 2],
            # diff_coeffs=(),
            duplicates=5)
            # duplicates=20)

        binn.to('cpu')

        parameters = binn.parameters()

        opt = torch.optim.Adam(parameters, lr=0.001)

        model = model_wrapper(
            model=binn,
            optimizer=opt,
            loss=binn.loss,
            dir_name=dir_name,
            save_name=f'{dir_name}/binn')
                
        model.load(f"{dir_name}/binn_best_val_model", device='cpu')
        # model.model.prune(thresh=5)
        # Print equation
        fn = f'{dir_name}/equation.txt'
        for term in model.model.generate_equation():
            print(f'{term}')
        
        model.model.fine_tune_eql(threshold=0.01, epsilon=0.1)
        for term in model.model.generate_equation():
            print(f'{term}')
            
        if not model.model.diff_coeffs:          
            print(f'{[D.item() for D in model.model.diffusion_fitter()]}\n')



Directory: binn_eql_batch_50000_points_100_repeat_2
Validation Loss: 0.018496
-0.511 * u
-0.746 * u
0.853 * v * u^1.843 / (1 + 0.010 * u^1.843)
0.196 * u * (1 / 0.350 - v^4.000 / (1 + 0.350 * v^4.000))
Simplifying Hill 64 -> Poly 0
  Error: 0.0794, Multiplier: 2.9230
Fine-tuning committed.
-0.684 * u
0.853 * v * u^1.843 / (1 + 0.010 * u^1.843)
[0.0061239516362547874, 0.6178156733512878]


Directory: binn_eql_batch_50000_points_100_repeat_1
Validation Loss: 0.01916
-0.633 * u
0.686 * v * u^1.941 / (1 + 0.010 * u^1.941)
Fine-tuning committed.
-0.633 * u
0.686 * v * u^1.941 / (1 + 0.010 * u^1.941)
[0.005975507665425539, 0.5981028079986572]


Directory: binn_eql_batch_50000_points_100_repeat_3
Validation Loss: 0.0218
0.118 * v * u^2.562 / (1 + 0.003 * u^2.562)
0.364 * v * u^2.334 / (1 + 0.044 * u^2.334)
-0.335 * u * (1 / 0.504 - v^1.001 / (1 + 0.504 * v^1.001))
Merging Duplicate Hills: 43 and 59 (Dist: 0.0013)
Fine-tuning committed.
0.482 * v * u^2.448 / (1 + 0.011 * u^2.448)
-0.335 * u *